In [ ]:
# importing all the required dependencies
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE # to handle imbalanced
from collections import Counter
# ml algorithms
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score
from sklearn.metrics import precision_score, f1_score, recall_score
import pickle # to save the model
# to ignore warnings
import warnings
warnings.filterwarnings('ignore')

In [2]:
data = pd.read_csv('../data/ml_data.csv')
data.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,Churn
0,Female,0,Yes,No,1,No,No,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,Yes
3,Male,0,No,No,45,No,No,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer,42.30,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,Yes


In [3]:
data.shape

(7016, 19)

In [4]:
data.duplicated().sum()

np.int64(0)

In [5]:
# separate features and target
features = data.drop(columns='Churn')
target = data[['Churn']]

# split train and test set
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state= 63)

In [6]:
y_train.value_counts(normalize=True)

Churn
No       0.733963
Yes      0.266037
Name: proportion, dtype: float64

In [7]:
data.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'Churn'],
      dtype='object')

In [8]:
# initialize a dictionary to store the encoder and column
encoders = {}
# List of columns to which label encoding was applied
label_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies','PaperlessBilling','PaymentMethod', 'Contract']

In [9]:
# label encoding

for col in label_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    X_test[col] = le.transform(X_test[col])
    encoders[col] = le

In [10]:
# mapping the target feature into numeric 
y_train['Churn'] = y_train['Churn'].map({'No':0, 'Yes':1})
y_test['Churn'] = y_test['Churn'].map({'No':0, 'Yes':1})

In [11]:
# save the encoders 
encoder_name = '../models/encoder.pkl'
with open(encoder_name, 'wb') as f:
    pickle.dump(encoders, f)

- We are going to apply tree-based algorithms for modelling, so not scaling the numerical features.

- To balance classes, we are using an over-sampling technique called SMOTE.
- Class imbalanced ratio is = (1: 2.77).

In [12]:
X_train.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges
6705,0,0,1,0,72,1,1,1,1,1,1,1,1,1,2,0,2,115.15
975,1,0,0,0,1,1,0,1,0,0,0,0,0,0,0,1,2,69.90
53,0,1,1,0,8,1,1,1,0,1,0,0,0,0,0,1,1,80.65
4115,1,0,1,1,6,1,0,2,0,0,0,0,0,0,0,0,3,19.55
1622,0,0,1,0,9,1,0,1,0,0,1,0,0,0,0,1,0,75.75


In [13]:
# smote 
sm = SMOTE(sampling_strategy='auto', random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print(y_train.value_counts())
print(y_train_res.value_counts())

Churn
0        4119
1        1493
Name: count, dtype: int64
Churn
0        4119
1        4119
Name: count, dtype: int64


In [14]:
# models to train 
models = {
    'Decision Tree' : DecisionTreeClassifier(random_state=56),
    'Random For' : RandomForestClassifier( random_state=63),
    'Xgboost' : xgb.XGBClassifier(random_state= 42),
    'Gradient Boost' : GradientBoostingClassifier(random_state=42),
    'LGBM' : lgb.LGBMClassifier(random_state=54)
}

In [15]:
# iterate through each model, train them and store their accuracy, avg accuracy, precision and recall
results = []
for model_name, model in models.items():
    model.fit(X_train_res, y_train_res)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    scores = cross_val_score(model, X_train_res, y_train_res, cv=5, scoring='accuracy')
    avg_accuracy = scores.mean()
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    results.append({
        'Model' : model_name, 
        'Accuracy' : accuracy, 
        'Avg Accuracy' : avg_accuracy,
        'Precision' : precision,
        'Recall' : recall})
    
result_df = pd.DataFrame(results)

[LightGBM] [Info] Number of positive: 4119, number of negative: 4119
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000873 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 364
[LightGBM] [Info] Number of data points in the train set: 8238, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Number of positive: 3295, number of negative: 3295
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000757 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 364
[LightGBM] [Info] Number of data points in the train set: 6590, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[L

In [16]:
result_df

,Model,Accuracy,Avg Accuracy,Precision,Recall
0,Decision Tree,0.717949,0.783697,0.462085,0.535714
1,Random For,0.778490,0.832740,0.572207,0.576923
2,Xgboost,0.774929,0.833470,0.560302,0.612637
3,Gradient Boost,0.777066,0.814043,0.554140,0.717033
4,LGBM,0.779202,0.826549,0.564593,0.648352


- The primary goal is to identify customers who are likely to churn, so give more importance to false negatives.
- As we can see, the Gradient Boosting algorithm performs better than the other algorithms in terms of recall, so we will select it as our final model.

In [17]:
classifier = GradientBoostingClassifier(random_state=42)

classifier.fit(X_train_res, y_train_res)
y_pred = classifier.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[830 210]
 [103 261]]
              precision    recall  f1-score   support

           0       0.89      0.80      0.84      1040
           1       0.55      0.72      0.63       364

    accuracy                           0.78      1404
   macro avg       0.72      0.76      0.73      1404
weighted avg       0.80      0.78      0.79      1404



In [18]:
# save the gradient model
filename = '../models/gradient_model.pkl'
with open(filename, 'wb') as f:
    pickle.dump(classifier, f)